In [ ]:
#Referenz ⇒ Canonical Form 
#pdb ID in der pdb suchen ⇒ FASTA sequence file downloaden  
#bei SAbPred: SCALOP in Submission form die Datei hochladen 
#results: für jedes CDR (H1, H2, L1, L2, L3) erkennt er CDR Sequenz aus der gesamten Sequenz (mit Antibody und antigen) und gibt einem canonical form und median structure 
#(L1-11-A → Canonical form A für eine L1-Schleife mit 11 Aminosäuren)
#muss man für alle unsere pdb Einträge machen und dann canonical cluster nochmal im code definieren (also die pdb Einträge zuordnen)
#dann der vergleich mit V-measure

In [1]:
#fasta dateien downloaden

import requests      #Modul zum Herunterladen von Daten aus dem Internet
import os            #Modul für Dateipfade und Ordnerverwaltung

def download_fasta(pdb_id, outdir="fasta_files"):
    """
    Lädt die FASTA-Sequenzdatei für einen gegebenen PDB-Eintrag
    von der RCSB PDB-Website herunter und speichert sie lokal.

    Parameter:
    - pdb_id: z.B. "1abc" (Groß-/Kleinschreibung egal)
    - outdir: Zielordner, in dem die FASTA-Dateien gespeichert werden

    Rückgabe:
    - Pfad zur gespeicherten FASTA-Datei (oder None bei Fehler)
    """

    #URL zur FASTA-Datei auf der rcsb.org-Website (liefert alle Chains)
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display"

    #HTTP-GET-Request an die URL schicken
    response = requests.get(url)

    #Prüfen ob der Download erfolgreich war (Statuscode 200 = OK)
    if response.status_code == 200:

        #Zielordner anlegen, falls er noch nicht existiert
        os.makedirs(outdir, exist_ok=True)

        #Speicherpfad für die Datei zusammensetzen
        fasta_path = os.path.join(outdir, f"{pdb_id}.fasta")

        #Inhalt in Datei schreiben
        with open(fasta_path, "w") as f:
            f.write(response.text)

        #Pfad zur fertigen Datei zurückgeben
        return fasta_path

    else:
        #Fehlerausgabe, falls Download fehlgeschlagen
        print(f"Fehler beim Herunterladen von {pdb_id} (Status: {response.status_code})")
        return None

In [ ]:
#hochladen auf sabpred und extraktion der canonical forms

In [ ]:
import time
import pandas as pd

# Automatisierung mit Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Hilft, automatisch passenden ChromeDriver zu finden
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
#pdb ids aus unerer geflterten datei laden
df = pd.read_csv("ab_ag_uniquesequences.csv")  
pdb_ids = df["pdb"].dropna().unique() #nur eindeutige, nicht leer

#automatisierten brwoser starten
# Chrome-Optionen: "headless" = läuft ohne sichtbares Fenster
options = webdriver.ChromeOptions()
options.add_argument('--headless')  # unsichtbar im Hintergrund

# Browser starten mit automatisch installiertem Treiber (=Schnittstelle zwischen python skript und browser)
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

results = []  # Hier wird jedes Canonical-Form-Ergebnis als dict gespeichert

#Iteration über PDB-IDs
for pdb_id in pdb_ids:
    pdb_id = pdb_id.lower()  # SCALOP erwartet kleingeschriebene IDs

    # Gehe zur SCALOP-Webseite
    driver.get("https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred/scalop")

    # Finde das Eingabefeld und trage die PDB-ID ein
    input_box = driver.find_element(By.NAME, "pdb")
    input_box.clear()
    input_box.send_keys(pdb_id)

    # Finde und klicke den "Submit"-Button
    submit_button = driver.find_element(By.XPATH, '//input[@type="submit" and @value="Submit"]')
    submit_button.click()

    #auf ergebnisse warten
    try:
    # Warten bis Ergebnisse (Tabelle) erscheinen (max. 10 Sekunden)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//table')))

    # Alle Zeilen (außer Header) in der Ergebnistabelle erfassen
        rows = driver.find_elements(By.XPATH, '//table//tr[position()>1]')      

        #für jede zeile relevante infos extrahieren
        for row in rows:
            cols = row.find_elements(By.TAG_NAME, 'td')
            if len(cols) >= 5:
                chain = cols[0].text.strip()
                cdr = cols[1].text.strip()
                length = cols[2].text.strip()
                canonical_form = cols[3].text.strip()

                # Ergebnis in Dictionary speichern
                results.append({
                "PDB_ID": pdb_id,
                "Chain": chain,
                "CDR": cdr,
                "Length": length,
                "Canonical_Form": canonical_form })

    except Exception as e:
            print(f"Fehler bei {pdb_id}: {e}")
            continue  # Wenn etwas schiefläuft, einfach zur nächsten PDB-ID springen

#browser schließen und ergbnisse speichern
# Browser schließen
driver.quit()

# Ergebnisse in DataFrame umwandeln und abspeichern
df_results = pd.DataFrame(results)
df_results.to_csv("scalop_canonical_forms.csv", index=False)